In [3]:
import json

# Definimos las celdas del notebook
cells = [
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["# 🚗 Homework 2 - Car Fuel Efficiency\n", "Ejecuta este notebook celda por celda. Las respuestas se imprimirán al final de cada bloque."]
    },
    {
        "cell_type": "code",
        "metadata": {},
        "source": [
            "import pandas as pd\n",
            "import numpy as np\n",
            "\n",
            "# 0. CARGA Y PREPARACIÓN DEL DATASET\n",
            "url = \"https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv\"\n",
            "df = pd.read_csv(url)\n",
            "\n",
            "cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']\n",
            "df = df[cols]\n",
            "\n",
            "# QUESTION 1\n",
            "missing_col = df.isnull().sum()[df.isnull().sum() > 0].index.tolist()[0]\n",
            "print(f\"Q1 - Columna con missing values: '{missing_col}'\")\n",
            "\n",
            "# QUESTION 2\n",
            "median_hp = df['horsepower'].median()\n",
            "print(f\"Q2 - Mediana de horsepower: {median_hp}\")"
        ]
    },
    {
        "cell_type": "code",
        "metadata": {},
        "source": [
            "# ==========================================\n",
            "# FUNCIONES AUXILIARES (Regresión Lineal)\n",
            "# ==========================================\n",
            "def train_linear_regression(X, y):\n",
            "    ones = np.ones(X.shape[0])\n",
            "    X = np.column_stack([ones, X])\n",
            "    XTX = X.T.dot(X)\n",
            "    XTX_inv = np.linalg.inv(XTX)\n",
            "    w = XTX_inv.dot(X.T).dot(y)\n",
            "    return w[0], w[1:]\n",
            "\n",
            "def train_linear_regression_reg(X, y, r=0.0):\n",
            "    ones = np.ones(X.shape[0])\n",
            "    X = np.column_stack([ones, X])\n",
            "    XTX = X.T.dot(X)\n",
            "    reg = r * np.eye(XTX.shape[0])\n",
            "    reg[0, 0] = 0  # No regularizar el bias\n",
            "    XTX = XTX + reg\n",
            "    XTX_inv = np.linalg.inv(XTX)\n",
            "    w = XTX_inv.dot(X.T).dot(y)\n",
            "    return w[0], w[1:]\n",
            "\n",
            "def predict(X, w0, w):\n",
            "    ones = np.ones(X.shape[0])\n",
            "    X = np.column_stack([ones, X])\n",
            "    return X.dot(np.concatenate([[w0], w]))\n",
            "\n",
            "def rmse(y_true, y_pred):\n",
            "    return np.sqrt(np.mean((y_true - y_pred) ** 2))\n",
            "\n",
            "def get_splits(seed=42):\n",
            "    n = len(df)\n",
            "    n_val = int(n * 0.2)\n",
            "    n_test = int(n * 0.2)\n",
            "    n_train = n - n_val - n_test\n",
            "    \n",
            "    np.random.seed(seed)\n",
            "    idx = np.arange(n)\n",
            "    np.random.shuffle(idx)\n",
            "    \n",
            "    df_train = df.iloc[idx[:n_train]].copy()\n",
            "    df_val = df.iloc[idx[n_train:n_train + n_val]].copy()\n",
            "    df_test = df.iloc[idx[n_train + n_val:]].copy()\n",
            "    return df_train, df_val, df_test\n",
            "\n",
            "target_col = 'horsepower'"
        ]
    },
    {
        "cell_type": "code",
        "metadata": {},
        "source": [
            "# ==========================================\n",
            "# QUESTION 3: Imputación (0 vs Media)\n",
            "# ==========================================\n",
            "df_train, df_val, df_test = get_splits(seed=42)\n",
            "\n",
            "# Opción A: Rellenar con 0\n",
            "df_tr_0, df_va_0 = df_train.copy(), df_val.copy()\n",
            "df_tr_0[target_col] = df_tr_0[target_col].fillna(0)\n",
            "df_va_0[target_col] = df_va_0[target_col].fillna(0)\n",
            "\n",
            "X_tr_0 = df_tr_0.drop(columns=['fuel_efficiency_mpg']).values\n",
            "y_tr_0 = df_tr_0['fuel_efficiency_mpg'].values\n",
            "X_va_0 = df_va_0.drop(columns=['fuel_efficiency_mpg']).values\n",
            "y_va_0 = df_va_0['fuel_efficiency_mpg'].values\n",
            "\n",
            "w0_0, w_0 = train_linear_regression(X_tr_0, y_tr_0)\n",
            "rmse_0 = round(rmse(y_va_0, predict(X_va_0, w0_0, w_0)), 3)\n",
            "\n",
            "# Opción B: Rellenar con la media\n",
            "mean_val = df_train[target_col].mean()\n",
            "df_tr_m, df_va_m = df_train.copy(), df_val.copy()\n",
            "df_tr_m[target_col] = df_tr_m[target_col].fillna(mean_val)\n",
            "df_va_m[target_col] = df_va_m[target_col].fillna(mean_val)\n",
            "\n",
            "X_tr_m = df_tr_m.drop(columns=['fuel_efficiency_mpg']).values\n",
            "y_tr_m = df_tr_m['fuel_efficiency_mpg'].values\n",
            "X_va_m = df_va_m.drop(columns=['fuel_efficiency_mpg']).values\n",
            "y_va_m = df_va_m['fuel_efficiency_mpg'].values\n",
            "\n",
            "w0_m, w_m = train_linear_regression(X_tr_m, y_tr_m)\n",
            "rmse_m = round(rmse(y_va_m, predict(X_va_m, w0_m, w_m)), 3)\n",
            "\n",
            "print(f\"Q3 - RMSE con 0: {rmse_0} | RMSE con media: {rmse_m}\")\n",
            "print(f\"-> Mejor opción: {'With mean' if rmse_m < rmse_0 else 'With 0'}\")"
        ]
    },
    {
        "cell_type": "code",
        "metadata": {},
        "source": [
            "# ==========================================\n",
            "# QUESTION 4: Regularización (diferentes 'r')\n",
            "# ==========================================\n",
            "df_train, df_val, df_test = get_splits(seed=42)\n",
            "df_train[target_col] = df_train[target_col].fillna(0)\n",
            "df_val[target_col] = df_val[target_col].fillna(0)\n",
            "\n",
            "X_tr = df_train.drop(columns=['fuel_efficiency_mpg']).values\n",
            "y_tr = df_train['fuel_efficiency_mpg'].values\n",
            "X_va = df_val.drop(columns=['fuel_efficiency_mpg']).values\n",
            "y_va = df_val['fuel_efficiency_mpg'].values\n",
            "\n",
            "r_values = [0, 0.01, 0.1, 1, 5, 10, 100]\n",
            "rmse_r = {}\n",
            "\n",
            "for r in r_values:\n",
            "    w0, w = train_linear_regression_reg(X_tr, y_tr, r=r)\n",
            "    rmse_r[r] = round(rmse(y_va, predict(X_va, w0, w)), 4)\n",
            "\n",
            "best_r = min(rmse_r, key=rmse_r.get)\n",
            "print(f\"Q4 - RMSE por r: {rmse_r}\")\n",
            "print(f\"-> Mejor r: {best_r}\")"
        ]
    },
    {
        "cell_type": "code",
        "metadata": {},
        "source": [
            "# ==========================================\n",
            "# QUESTION 5: Estabilidad del modelo (seeds)\n",
            "# ==========================================\n",
            "seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]\n",
            "rmse_scores = []\n",
            "\n",
            "for seed in seeds:\n",
            "    df_tr, df_va, _ = get_splits(seed=seed)\n",
            "    df_tr[target_col] = df_tr[target_col].fillna(0)\n",
            "    df_va[target_col] = df_va[target_col].fillna(0)\n",
            "    \n",
            "    X_tr = df_tr.drop(columns=['fuel_efficiency_mpg']).values\n",
            "    y_tr = df_tr['fuel_efficiency_mpg'].values\n",
            "    X_va = df_va.drop(columns=['fuel_efficiency_mpg']).values\n",
            "    y_va = df_va['fuel_efficiency_mpg'].values\n",
            "    \n",
            "    w0, w = train_linear_regression(X_tr, y_tr)\n",
            "    rmse_scores.append(rmse(y_va, predict(X_va, w0, w)))\n",
            "\n",
            "std_rmse = round(np.std(rmse_scores), 3)\n",
            "print(f\"Q5 - Desviación estándar de los RMSE: {std_rmse}\")"
        ]
    },
    {
        "cell_type": "code",
        "metadata": {},
        "source": [
            "# ==========================================\n",
            "# QUESTION 6: RMSE en Test (seed 9, r=0.001)\n",
            "# ==========================================\n",
            "df_train, df_val, df_test = get_splits(seed=9)\n",
            "\n",
            "df_train_val = pd.concat([df_train, df_val])\n",
            "df_train_val[target_col] = df_train_val[target_col].fillna(0)\n",
            "df_test[target_col] = df_test[target_col].fillna(0)\n",
            "\n",
            "X_tr_val = df_train_val.drop(columns=['fuel_efficiency_mpg']).values\n",
            "y_tr_val = df_train_val['fuel_efficiency_mpg'].values\n",
            "X_te = df_test.drop(columns=['fuel_efficiency_mpg']).values\n",
            "y_te = df_test['fuel_efficiency_mpg'].values\n",
            "\n",
            "w0_6, w_6 = train_linear_regression_reg(X_tr_val, y_tr_val, r=0.001)\n",
            "rmse_test = round(rmse(y_te, predict(X_te, w0_6, w_6)), 3)\n",
            "\n",
            "print(f\"Q6 - RMSE en el dataset de Test: {rmse_test}\")"
        ]
    }
]

# Crear la estructura del notebook
notebook = {
    "cells": cells,
    "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}},
    "nbformat": 4,
    "nbformat_minor": 4
}

# Guardar el archivo
with open("homework2.ipynb", "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=1)

print("✅ ¡LISTO! El archivo 'homework2.ipynb' se ha creado en tu carpeta actual.")
print("📂 Ábrelo directamente desde Jupyter / VS Code y ejecuta las celdas.")

✅ ¡LISTO! El archivo 'homework2.ipynb' se ha creado en tu carpeta actual.
📂 Ábrelo directamente desde Jupyter / VS Code y ejecuta las celdas.
